# 03-2: XGBoost モデル

M5 Forecasting - Accuracy コンペティション

- **モデル**: XGBoost (Gradient Boosting)
- **評価指標**: RMSE / RMSLE
- **NaN戦略**: NaN(デフォルト) — XGBoostはNaN対応
- **CV戦略**: 時系列5-fold CV（28日window、全モデル統一）

In [ ]:
import sys
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xgboost as xgb
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.unicode_minus'] = False

sys.path.append('../../..')
sys.path.append('../..')
from src.features import build_features, prepare_train_data, get_val_folds, get_feature_cols

INTERMEDIATE_DIR = Path('./intermediate')

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('セットアップ完了')

## 1. データ読み込みと特徴量生成

In [ ]:
df, feature_cols, val_folds = build_features(use_cache=True)
df_train = prepare_train_data(df, feature_cols)
print(f'特徴量数: {len(feature_cols)}')

## 2. XGBoost パラメータ設定

In [ ]:
xgb_params = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.1,
    'eval_metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 8,
    'min_child_weight': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'tree_method': 'hist',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
}
n_estimators = 2000

print('XGBoost パラメータ:')
for k, v in xgb_params.items():
    print(f'  {k}: {v}')
print(f'  n_estimators: {n_estimators}')

## 3. 時系列CV評価

In [ ]:
cv_scores = []
best_iterations = []
importances = []

for fold_info in val_folds:
    fold = fold_info['fold']
    val_start = pd.Timestamp(fold_info['val_start'])
    val_end = pd.Timestamp(fold_info['val_end'])
    
    train_mask = df_train['date'] < val_start
    valid_mask = (df_train['date'] >= val_start) & (df_train['date'] <= val_end)
    
    X_train = df_train.loc[train_mask, feature_cols]
    y_train = df_train.loc[train_mask, 'sales']
    X_valid = df_train.loc[valid_mask, feature_cols]
    y_valid = df_train.loc[valid_mask, 'sales']
    
    print(f'\n--- Fold {fold} ---')
    print(f'  Train: {X_train.shape[0]:,} rows, Valid: {X_valid.shape[0]:,} rows')
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=0,
    )
    
    pred = model.predict(dvalid)
    pred = np.clip(pred, 0, None)
    score = rmse(y_valid, pred)
    score_rmsle = rmsle(y_valid, pred)
    
    cv_scores.append({
        'fold': fold, 'rmse': score, 'rmsle': score_rmsle,
        'best_iter': model.best_iteration,
    })
    best_iterations.append(model.best_iteration)
    
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': [model.get_score(importance_type='gain').get(f, 0) for f in feature_cols],
    })
    imp['fold'] = fold
    importances.append(imp)
    
    print(f'  RMSE: {score:.4f}, RMSLE: {score_rmsle:.5f}, best_iter: {model.best_iteration}')

cv_df = pd.DataFrame(cv_scores)
print(f'\n=== CV結果サマリー ===')
print(f'RMSE  平均: {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')
print(f'RMSLE 平均: {cv_df["rmsle"].mean():.5f} ± {cv_df["rmsle"].std():.5f}')
print(cv_df.to_string(index=False))

## 4. 特徴量重要度

In [ ]:
imp_all = pd.concat(importances)
imp_mean = imp_all.groupby('feature')['importance'].mean().sort_values(ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_mean['feature'][::-1], imp_mean['importance'][::-1], color='coral')
ax.set_title('XGBoost 特徴量重要度（Gain平均）')
ax.set_xlabel('平均Gain')
plt.tight_layout()
plt.show()

## 5. 全データ再学習 & 中間データ保存

In [ ]:
best_iter = int(np.mean(best_iterations))
print(f'全データ再学習: iterations = {best_iter}')

X_all = df_train[feature_cols]
y_all = df_train['sales']

dtrain_all = xgb.DMatrix(X_all, label=y_all)
final_model = xgb.train(xgb_params, dtrain_all, num_boost_round=best_iter)

# V4 & V11
train_pred = np.clip(final_model.predict(dtrain_all), 0, None)
train_rmse = rmse(y_all, train_pred)
cv_rmse = cv_df['rmse'].mean()
overfit_ratio = train_rmse / cv_rmse
print(f'Train RMSE: {train_rmse:.4f}, CV RMSE: {cv_rmse:.4f}, 過学習比率: {overfit_ratio:.3f}')

# 保存
results = {
    'model_name': 'XGBoost',
    'params': xgb_params,
    'cv_scores': cv_df.to_dict('records'),
    'cv_mean_rmse': cv_df['rmse'].mean(),
    'cv_std_rmse': cv_df['rmse'].std(),
    'cv_mean_rmsle': cv_df['rmsle'].mean(),
    'cv_std_rmsle': cv_df['rmsle'].std(),
    'best_iteration': best_iter,
    'feature_importance': imp_mean.to_dict('records'),
    'overfit_ratio': overfit_ratio,
}

with open(INTERMEDIATE_DIR / '03-2_xgb_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print('03-2_xgb_results.pkl を保存しました')
print(f'\n最終結果: CV RMSE = {cv_df["rmse"].mean():.4f} ± {cv_df["rmse"].std():.4f}')